In [4]:
import pandas as pd
import numpy as np


In [5]:
sales = pd.read_csv("../data/raw/sales_transactions.csv")

In [6]:
sales.head()

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id
0,2025-04-02,RCPT00000001,ST16,SKU02498,CUST01410,1,2379.11,2379.11,In-Store,0.0,NaN
1,2025-04-02,RCPT00000001,ST16,SKU04596,CUST01410,3,335.55,1006.65,In-Store,0.0,NaN
2,2022-04-24,RCPT00000002,ST15,SKU00078,CUST00134,1,820.53,820.53,Online,0.0,NaN
3,2022-04-24,RCPT00000002,ST15,SKU00554,CUST00134,2,88.32,176.64,Online,0.0,NaN
4,2024-09-22,RCPT00000003,ST20,SKU03727,CUST08826,1,1660.93,1660.93,Online,0.0,NaN


In [7]:
sales.tail()

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id
9945506,2023-09-09,RCPT05181381,ST04,SKU00701,CUST06397,2,39.24,68.51,In-Store,12.7,PROMO010
9945507,2025-12-03,RCPT05181382,ST03,SKU02163,CUST04945,1,791.70,791.70,In-Store,0.0,NaN
9945508,2025-12-03,RCPT05181382,ST03,SKU02866,CUST04945,4,94.26,377.04,In-Store,0.0,NaN
9945509,2023-01-20,RCPT05181383,ST03,SKU03727,CUST05244,1,1660.93,1660.93,In-Store,0.0,NaN
9945510,2023-07-01,RCPT05181384,ST20,SKU04596,CUST08714,1,335.55,335.55,In-Store,0.0,NaN


In [8]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9945511 entries, 0 to 9945510
Data columns (total 11 columns):
 #   Column        Dtype  
---  ------        -----  
 0   date          object 
 1   receipt_id    object 
 2   store_id      object 
 3   sku_id        object 
 4   customer_id   object 
 5   quantity      int64  
 6   unit_price    float64
 7   total_value   float64
 8   channel       object 
 9   discount_pct  float64
 10  promo_id      object 
dtypes: float64(3), int64(1), object(7)
memory usage: 834.7+ MB


In [9]:
sales.describe()

,quantity,unit_price,total_value,discount_pct
count,9.945511e+06,9.945511e+06,9.945511e+06,9.945511e+06
mean,1.880336e+00,6.215749e+02,1.094512e+03,6.289965e+00
std,1.089063e+00,6.751952e+02,1.536814e+03,1.383267e+01
min,1.000000e+00,2.441000e+01,1.225000e+01,0.000000e+00
25%,1.000000e+00,1.734300e+02,2.351700e+02,0.000000e+00
50%,2.000000e+00,3.367200e+02,5.432000e+02,0.000000e+00
75%,3.000000e+00,9.256700e+02,1.277610e+03,0.000000e+00
max,5.000000e+00,4.755470e+03,2.377735e+04,4.980000e+01


In [10]:
sales.isnull().sum()

date                  0
receipt_id            0
store_id              0
sku_id                0
customer_id           0
quantity              0
unit_price            0
total_value           0
channel               0
discount_pct          0
promo_id        7855345
dtype: int64

In [11]:
sales.nunique()

date               1461
receipt_id      5167864
store_id             30
sku_id             5000
customer_id       10000
quantity              5
unit_price         4837
total_value      182745
channel               3
discount_pct         84
promo_id             96
dtype: int64

In [12]:
sales["channel"].value_counts(normalize=True)*100

channel
In-Store      55.269277
Online        29.743560
Mobile App    14.987164
Name: proportion, dtype: float64

In [13]:
sales["quantity"].value_counts()

quantity
1    4971513
2    2486694
3    1491849
4     696817
5     298638
Name: count, dtype: int64

In [14]:
sales["discount_pct"].value_counts().sort_index().head(15)

discount_pct
0.0     7855345
5.0          27
5.8        3695
5.9       11035
6.8        3994
7.0       67557
7.9        8646
8.1        2295
8.5       15013
8.6       16784
9.0       77458
9.7       13730
9.9       15984
10.2       6156
10.3       1880
Name: count, dtype: int64

In [15]:
# check business perspective
(sales["quantity"]*sales["unit_price"]==sales["total_value"]).sum()

np.int64(7375195)

In [16]:
# value doesnot match with 0.0 per discount
diff=sales["quantity"]*sales["unit_price"]-sales["total_value"]
diff.describe()

count    9.945511e+06
mean     7.431736e+01
std      2.953767e+02
min     -3.637979e-12
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.044346e+04
dtype: float64

In [17]:
diff.eq(0).sum()

np.int64(7375195)

In [18]:
diff.ne(0).sum()

np.int64(2570316)

In [19]:
mismatch=diff.ne(0)

sales.loc[mismatch,["quantity","unit_price","discount_pct","promo_id","total_value"]].head()

,quantity,unit_price,discount_pct,promo_id,total_value
1,3,335.55,0.0,NaN,1006.65
10,3,961.05,0.0,NaN,2883.15
15,2,233.64,49.7,PROMO030,235.04
19,1,1000.23,48.6,PROMO037,514.12
20,3,602.13,0.0,NaN,1806.39


In [20]:
diff.abs().describe()

count    9.945511e+06
mean     7.431736e+01
std      2.953767e+02
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      5.684342e-14
max      1.044346e+04
dtype: float64

In [21]:
diff.abs().sort_values(ascending=True).head(10)

8212561    0.0
9945489    0.0
9945490    0.0
9945491    0.0
9945492    0.0
9945493    0.0
9945494    0.0
9945495    0.0
9945496    0.0
9945497    0.0
dtype: float64

In [22]:
(diff.abs().gt(1e-8).sum())

np.int64(2090166)

In [23]:
expected_total=(sales["quantity"]*sales["unit_price"]*(1-sales["discount_pct"]/100))
expected_total.head()

0    2379.11
1    1006.65
2     820.53
3     176.64
4    1660.93
dtype: float64

In [24]:
(expected_total-sales["total_value"]).abs().describe()

count    9.945511e+06
mean     5.281566e-04
std      1.219890e-03
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      5.684342e-14
max      5.000000e-03
dtype: float64

In [25]:
sales.duplicated().sum()

np.int64(12897)

In [26]:
sales[sales.duplicated(keep=False)].sort_values("receipt_id").head(20)

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id
270,2025-05-11,RCPT00000149,ST10,SKU02141,CUST09365,2,471.88,943.76,Online,0.0,NaN
272,2025-05-11,RCPT00000149,ST10,SKU02141,CUST09365,2,471.88,943.76,Online,0.0,NaN
314,2022-08-03,RCPT00000169,ST29,SKU04321,CUST07066,1,961.30,961.30,In-Store,0.0,NaN
315,2022-08-03,RCPT00000169,ST29,SKU04321,CUST07066,1,961.30,961.30,In-Store,0.0,NaN
544,2025-11-14,RCPT00000289,ST06,SKU04321,CUST02146,1,961.30,961.30,In-Store,0.0,NaN
545,2025-11-14,RCPT00000289,ST06,SKU04321,CUST02146,1,961.30,961.30,In-Store,0.0,NaN
2935,2024-06-22,RCPT00001531,ST25,SKU04321,CUST01969,1,961.30,744.05,Online,22.6,PROMO045
2936,2024-06-22,RCPT00001531,ST25,SKU04321,CUST01969,1,961.30,744.05,Online,22.6,PROMO045
5845,2025-10-09,RCPT00003040,ST13,SKU04321,CUST08409,1,961.30,961.30,Mobile App,0.0,NaN
5846,2025-10-09,RCPT00003040,ST13,SKU04321,CUST08409,1,961.30,961.30,Mobile App,0.0,NaN


In [27]:
sales[sales.duplicated(keep=False)]["receipt_id"].nunique()

12739

In [28]:
sales[sales.duplicated(keep=False)].groupby(list(sales.columns)).size().value_counts().sort_index()

2    2656
3      30
Name: count, dtype: int64

In [29]:
dup_rows=sales[sales.duplicated(keep=False)]
dup_rows.groupby(list(sales.columns),dropna=False).size().value_counts().sort_index()

2    12598
3      148
4        1
Name: count, dtype: int64

In [30]:
sales["date"].min(),sales["date"].max()

('2022-01-01', '2025-12-31')

In [31]:
pd.to_datetime(sales["date"]).nunique()

1461

In [32]:
dates=pd.to_datetime(sales["date"])
dates.max()-dates.min()

Timedelta('1460 days 00:00:00')

In [33]:
pd.date_range(start=dates.min(),end=dates.max()).difference(dates)

DatetimeIndex([], dtype='datetime64[ns]', freq='D')

In [34]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9945511 entries, 0 to 9945510
Data columns (total 11 columns):
 #   Column        Dtype  
---  ------        -----  
 0   date          object 
 1   receipt_id    object 
 2   store_id      object 
 3   sku_id        object 
 4   customer_id   object 
 5   quantity      int64  
 6   unit_price    float64
 7   total_value   float64
 8   channel       object 
 9   discount_pct  float64
 10  promo_id      object 
dtypes: float64(3), int64(1), object(7)
memory usage: 834.7+ MB


In [35]:
sales["channel"].nunique()

3

In [36]:
sales["store_id"].nunique()

30

In [37]:
sales["sku_id"].nunique(),sales["customer_id"].nunique()

(5000, 10000)

In [38]:
sales[["quantity","unit_price","total_value","discount_pct"]].describe()

,quantity,unit_price,total_value,discount_pct
count,9.945511e+06,9.945511e+06,9.945511e+06,9.945511e+06
mean,1.880336e+00,6.215749e+02,1.094512e+03,6.289965e+00
std,1.089063e+00,6.751952e+02,1.536814e+03,1.383267e+01
min,1.000000e+00,2.441000e+01,1.225000e+01,0.000000e+00
25%,1.000000e+00,1.734300e+02,2.351700e+02,0.000000e+00
50%,2.000000e+00,3.367200e+02,5.432000e+02,0.000000e+00
75%,3.000000e+00,9.256700e+02,1.277610e+03,0.000000e+00
max,5.000000e+00,4.755470e+03,2.377735e+04,4.980000e+01


In [39]:
sales[["channel"]].value_counts()

channel   
In-Store      5496812
Online        2958149
Mobile App    1490550
Name: count, dtype: int64

In [40]:
sales[["store_id", "sku_id", "customer_id", "promo_id"]].nunique()

store_id          30
sku_id          5000
customer_id    10000
promo_id          96
dtype: int64

In [41]:
sales["customer_id"].unique()

array(['CUST01410', 'CUST00134', 'CUST08826', ..., 'CUST07767',
       'CUST07626', 'CUST09371'], shape=(10000,), dtype=object)

In [42]:
sales[sales.duplicated(keep=False)].sort_values(
    ["receipt_id", "date"]
).head(20)

,date,receipt_id,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct,promo_id
270,2025-05-11,RCPT00000149,ST10,SKU02141,CUST09365,2,471.88,943.76,Online,0.0,NaN
272,2025-05-11,RCPT00000149,ST10,SKU02141,CUST09365,2,471.88,943.76,Online,0.0,NaN
314,2022-08-03,RCPT00000169,ST29,SKU04321,CUST07066,1,961.30,961.30,In-Store,0.0,NaN
315,2022-08-03,RCPT00000169,ST29,SKU04321,CUST07066,1,961.30,961.30,In-Store,0.0,NaN
544,2025-11-14,RCPT00000289,ST06,SKU04321,CUST02146,1,961.30,961.30,In-Store,0.0,NaN
545,2025-11-14,RCPT00000289,ST06,SKU04321,CUST02146,1,961.30,961.30,In-Store,0.0,NaN
2935,2024-06-22,RCPT00001531,ST25,SKU04321,CUST01969,1,961.30,744.05,Online,22.6,PROMO045
2936,2024-06-22,RCPT00001531,ST25,SKU04321,CUST01969,1,961.30,744.05,Online,22.6,PROMO045
5845,2025-10-09,RCPT00003040,ST13,SKU04321,CUST08409,1,961.30,961.30,Mobile App,0.0,NaN
5846,2025-10-09,RCPT00003040,ST13,SKU04321,CUST08409,1,961.30,961.30,Mobile App,0.0,NaN


In [43]:
sales_clean = sales.drop_duplicates()

sales.shape, sales_clean.shape

((9945511, 11), (9932614, 11))

In [44]:
sales_clean.duplicated().sum()

np.int64(0)

In [45]:
sales_clean["date"]=pd.to_datetime(sales_clean["date"])

C:\Users\aadi1\AppData\Local\Temp\ipykernel_26572\2190478579.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales_clean["date"]=pd.to_datetime(sales_clean["date"])


In [46]:
sales_clean["date"].dtype

dtype('<M8[ns]')

In [47]:
sales_clean=sales.drop_duplicates().copy()

In [48]:
sales_clean["date"] = pd.to_datetime(sales_clean["date"])

In [49]:
sales_clean.shape

(9932614, 11)

In [50]:
sales_clean.duplicated().sum()

np.int64(0)

In [51]:
sales.duplicated().sum()

np.int64(12897)

In [52]:
sales_clean.isna().sum()

date                  0
receipt_id            0
store_id              0
sku_id                0
customer_id           0
quantity              0
unit_price            0
total_value           0
channel               0
discount_pct          0
promo_id        7845164
dtype: int64

In [53]:
sales_clean.dtypes

date            datetime64[ns]
receipt_id              object
store_id                object
sku_id                  object
customer_id             object
quantity                 int64
unit_price             float64
total_value            float64
channel                 object
discount_pct           float64
promo_id                object
dtype: object

In [54]:
expected_total.describe()

count    9.945511e+06
mean     1.094512e+03
std      1.536814e+03
min      1.225382e+01
25%      2.351700e+02
50%      5.432000e+02
75%      1.277610e+03
max      2.377735e+04
dtype: float64

In [55]:
expected_value=(sales_clean["quantity"]*sales_clean["unit_price"]*(1-sales_clean["discount_pct"]/100))
(sales_clean["total_value"]-expected_value).abs().max()

np.float64(0.005000000001018634)

In [56]:
customer = pd.read_csv("../data/raw/customer_master.csv")
inventory = pd.read_csv("../data/raw/inventory_snapshot.csv")
promotions = pd.read_csv("../data/raw/promotions.csv")
sku_flags = pd.read_csv("../data/raw/sku_inventory_flags.csv")
sku = pd.read_csv("../data/raw/sku_master.csv")
store = pd.read_csv("../data/raw/store_master.csv")

In [57]:
customer.shape

(10000, 7)

In [58]:
for name,df in {
    "customer":customer,
    "sku":sku,
    "store":store,
    "promotions":promotions,
    "inventory":inventory,
    "sku_flags":sku_flags,
    "sales":sales,"sales_clean":sales_clean
}.items():
    print(f"\n{name}")
    for col in df.columns:
        print("-",col)


customer
- cust_id
- age
- gender
- city
- loyalty_segment
- preferred_channel
- registration_date

sku
- sku_id
- sku_name
- category
- subcategory
- unit_price
- cost_price
- brand

store
- store_id
- store_name
- city
- store_type
- opening_date

promotions
- promo_id
- promo_name
- start_date
- end_date
- discount_pct
- promo_type
- target_type
- target_value

inventory
- store_id
- sku_id
- stock_on_hand
- reorder_point
- safety_stock
- last_restock_date

sku_flags
- sku_id
- flag
- affected_stores
- window_start
- window_end
- notes

sales
- date
- receipt_id
- store_id
- sku_id
- customer_id
- quantity
- unit_price
- total_value
- channel
- discount_pct
- promo_id

sales_clean
- date
- receipt_id
- store_id
- sku_id
- customer_id
- quantity
- unit_price
- total_value
- channel
- discount_pct
- promo_id


In [59]:
sales_clean=sales_clean.merge(
    customer[
        [
            "cust_id","age","gender","city","loyalty_segment",
            "preferred_channel","registration_date"
        ]
    ].rename(columns={"cust_id":"customer_id"}),
    on="customer_id",
    how="left"
)

In [60]:
sales_clean.shape

(9932614, 17)

In [61]:
(sales_clean["unit_price"] == sales_clean["sku_id"].map(sku.set_index("sku_id")["unit_price"])).value_counts()

True    9932614
Name: count, dtype: int64

In [62]:
sales_clean=sales_clean.merge(sku[["sku_id","sku_name","category","subcategory","cost_price","brand"]],
                              on="sku_id",how="left")

In [63]:
sales_clean.shape

(9932614, 22)

In [64]:
for name,df in {
    "sales_clean":sales_clean
}.items():
    print(f"\n{name}")
    for col in df.columns:
        print("-",col)


sales_clean
- date
- receipt_id
- store_id
- sku_id
- customer_id
- quantity
- unit_price
- total_value
- channel
- discount_pct
- promo_id
- age
- gender
- city
- loyalty_segment
- preferred_channel
- registration_date
- sku_name
- category
- subcategory
- cost_price
- brand


In [65]:
sales_clean=sales_clean.merge(store[["store_id","store_name","city","store_type","opening_date"]]
                              .rename(columns={"city":"store_city"}),on="store_id",
                              how="left")

In [66]:
sales_clean=sales_clean.rename(columns={"city":"customer_city"})

In [67]:
for name,df in {
    "sales_clean":sales_clean
}.items():
    print(f"\n{name}")
    for col in df.columns:
        print("-",col)


sales_clean
- date
- receipt_id
- store_id
- sku_id
- customer_id
- quantity
- unit_price
- total_value
- channel
- discount_pct
- promo_id
- age
- gender
- customer_city
- loyalty_segment
- preferred_channel
- registration_date
- sku_name
- category
- subcategory
- cost_price
- brand
- store_name
- store_city
- store_type
- opening_date


In [68]:
sales_clean=sales_clean.merge(promotions[["promo_id","promo_name","start_date","end_date","promo_type","target_type","target_value"]],
                              on ="promo_id",
                              how="left")

In [69]:
inventory.shape

(26408, 6)

In [70]:
inventory.duplicated(subset=["store_id","sku_id"]).sum()

np.int64(0)

In [71]:
sales_clean=sales_clean.merge(inventory[["store_id","sku_id","stock_on_hand"
                                         ,"reorder_point","safety_stock","last_restock_date"]],
                                         on=["store_id","sku_id"],
                                         how="left")

In [72]:
sales_clean.shape

(9932614, 36)

In [73]:
sku_flags["flag"].value_counts(dropna=False)

flag
SLOW_MOVER       400
STOCKOUT_RISK    200
Name: count, dtype: int64

In [74]:
sku_flags.head(10)

,sku_id,flag,affected_stores,window_start,window_end,notes
0,SKU04321,STOCKOUT_RISK,ST22;ST26;ST23;ST15;ST12;ST28;ST01;ST27;ST08;S...,2025-11-06,2025-12-03,Top-selling SKU (by observed volume); injected...
1,SKU04596,STOCKOUT_RISK,ST12;ST09;ST21;ST23;ST07;ST24;ST01;ST28;ST14;S...,2025-10-25,2025-11-06,Top-selling SKU (by observed volume); injected...
2,SKU03727,STOCKOUT_RISK,ST19;ST14;ST28;ST29;ST02;ST07;ST17;ST08;ST05;ST16,2025-11-08,2025-11-29,Top-selling SKU (by observed volume); injected...
3,SKU04154,STOCKOUT_RISK,ST15;ST03;ST30;ST09;ST24;ST19;ST28;ST02;ST07;S...,2025-11-12,2025-11-26,Top-selling SKU (by observed volume); injected...
4,SKU00953,STOCKOUT_RISK,ST09;ST19;ST27;ST20;ST01;ST03;ST11;ST10;ST22;S...,2025-11-29,2025-12-25,Top-selling SKU (by observed volume); injected...
5,SKU00190,STOCKOUT_RISK,ST12;ST25;ST21;ST26;ST11;ST19;ST28;ST02;ST15;S...,2025-11-14,2025-12-03,Top-selling SKU (by observed volume); injected...
6,SKU02141,STOCKOUT_RISK,ST09;ST21;ST06;ST16;ST07;ST30;ST27;ST10;ST11;S...,2025-12-05,2025-12-30,Top-selling SKU (by observed volume); injected...
7,SKU00994,STOCKOUT_RISK,ST18;ST04;ST29;ST30;ST11;ST26;ST07;ST15;ST14;S...,2025-12-17,2025-12-28,Top-selling SKU (by observed volume); injected...
8,SKU03292,STOCKOUT_RISK,ST09;ST06;ST15;ST27;ST25;ST01;ST11;ST30;ST04;S...,2025-10-22,2025-11-10,Top-selling SKU (by observed volume); injected...
9,SKU01775,STOCKOUT_RISK,ST17;ST15;ST08;ST27;ST16;ST05;ST12;ST20;ST03;S...,2025-10-27,2025-11-07,Top-selling SKU (by observed volume); injected...


In [75]:
sku_flags["affected_stores"].str.split(";").head()

0    [ST22, ST26, ST23, ST15, ST12, ST28, ST01, ST2...
1    [ST12, ST09, ST21, ST23, ST07, ST24, ST01, ST2...
2    [ST19, ST14, ST28, ST29, ST02, ST07, ST17, ST0...
3    [ST15, ST03, ST30, ST09, ST24, ST19, ST28, ST0...
4    [ST09, ST19, ST27, ST20, ST01, ST03, ST11, ST1...
Name: affected_stores, dtype: object

In [76]:
sku_flags_expanded = sku_flags.assign(
    store_id=sku_flags["affected_stores"].str.split(";")
).explode("store_id")

In [77]:
sku_flags_expanded["window_start"] = pd.to_datetime(sku_flags_expanded["window_start"])
sku_flags_expanded["window_end"] = pd.to_datetime(sku_flags_expanded["window_end"])

In [78]:
sku_flags_expanded["store_id"].dtype

dtype('O')

In [79]:
sku_flags_expanded["store_id"] = sku_flags_expanded["store_id"].astype(str)

In [80]:
sku_flags_match = sku_flags_expanded[
    ["sku_id", "store_id", "flag", "window_start", "window_end", "notes"]
].copy()

In [81]:
sales_flag_base = sales_clean[
    ["sku_id", "store_id", "date"]
].copy()

In [82]:
sales_flag_match = sales_flag_base.merge(
    sku_flags_match,
    on=["sku_id", "store_id"],
    how="left"
)

In [83]:
sales_flag_match = sales_flag_match[
    (sales_flag_match["date"] >= sales_flag_match["window_start"]) &
    (sales_flag_match["date"] <= sales_flag_match["window_end"])
]

In [84]:
sales_flag_match.groupby(
    ["sku_id", "store_id", "date"]
).size().value_counts()

Series([], Name: count, dtype: int64)

In [85]:
sales_flag_match.shape

(0, 7)

In [86]:
sales_flag_base.merge(
    sku_flags_match[["sku_id", "store_id"]],
    on=["sku_id", "store_id"],
    how="inner"
).shape

(1722653, 3)

In [87]:
sales_clean["date"].agg(["min", "max"]), sku_flags_match[["window_start", "window_end"]].agg(["min", "max"])

(min   2022-01-01
 max   2025-12-31
 Name: date, dtype: datetime64[ns],
     window_start window_end
 min   2025-10-17 2025-10-29
 max   2025-12-17 2025-12-31)

In [88]:
sales_flag_base[
    sales_flag_base["date"].between("2025-10-17", "2025-12-31")
].shape


(640397, 3)

In [89]:
sales_flag_base.describe


<bound method NDFrame.describe of            sku_id store_id       date
0        SKU02498     ST16 2025-04-02
1        SKU04596     ST16 2025-04-02
2        SKU00078     ST15 2022-04-24
3        SKU00554     ST15 2022-04-24
4        SKU03727     ST20 2024-09-22
...           ...      ...        ...
9932609  SKU00701     ST04 2023-09-09
9932610  SKU02163     ST03 2025-12-03
9932611  SKU02866     ST03 2025-12-03
9932612  SKU03727     ST03 2023-01-20
9932613  SKU04596     ST20 2023-07-01

[9932614 rows x 3 columns]>

In [90]:
sales_flag_base.merge(
    sku_flags_match,
    on=["sku_id", "store_id"],
    how="inner"
).head(10)


,sku_id,store_id,date,flag,window_start,window_end,notes
0,SKU01371,ST19,2025-05-20,STOCKOUT_RISK,2025-11-27,2025-12-24,Top-selling SKU (by observed volume); injected...
1,SKU00830,ST23,2022-11-19,STOCKOUT_RISK,2025-10-20,2025-10-30,Top-selling SKU (by observed volume); injected...
2,SKU04321,ST22,2022-09-29,STOCKOUT_RISK,2025-11-06,2025-12-03,Top-selling SKU (by observed volume); injected...
3,SKU02580,ST27,2024-10-09,SLOW_MOVER,NaT,NaT,Bottom-selling SKU (by observed volume); injec...
4,SKU00500,ST13,2024-05-29,STOCKOUT_RISK,2025-10-24,2025-11-16,Top-selling SKU (by observed volume); injected...
5,SKU04596,ST13,2024-05-29,STOCKOUT_RISK,2025-10-25,2025-11-06,Top-selling SKU (by observed volume); injected...
6,SKU02355,ST22,2022-08-24,STOCKOUT_RISK,2025-10-21,2025-11-01,Top-selling SKU (by observed volume); injected...
7,SKU04321,ST27,2022-03-24,STOCKOUT_RISK,2025-11-06,2025-12-03,Top-selling SKU (by observed volume); injected...
8,SKU02329,ST11,2024-12-16,SLOW_MOVER,NaT,NaT,Bottom-selling SKU (by observed volume); injec...
9,SKU00369,ST14,2025-07-11,STOCKOUT_RISK,2025-10-17,2025-11-04,Top-selling SKU (by observed volume); injected...


In [91]:
sales_clean.columns.tolist()

['date',
 'receipt_id',
 'store_id',
 'sku_id',
 'customer_id',
 'quantity',
 'unit_price',
 'total_value',
 'channel',
 'discount_pct',
 'promo_id',
 'age',
 'gender',
 'customer_city',
 'loyalty_segment',
 'preferred_channel',
 'registration_date',
 'sku_name',
 'category',
 'subcategory',
 'cost_price',
 'brand',
 'store_name',
 'store_city',
 'store_type',
 'opening_date',
 'promo_name',
 'start_date',
 'end_date',
 'promo_type',
 'target_type',
 'target_value',
 'stock_on_hand',
 'reorder_point',
 'safety_stock',
 'last_restock_date']

In [92]:
sales_clean["year"]=sales_clean["date"].dt.year

In [93]:
sales_clean["month"]=sales_clean["date"].dt.month

In [94]:
sales_clean["day_of_week"]=sales_clean["date"].dt.dayofweek

In [95]:
sales_clean["profit_per_unit"]=sales_clean["unit_price"]-sales_clean["cost_price"]

In [96]:
sales_clean["profit_margin_pct"]=(sales_clean["profit_per_unit"]/sales_clean["unit_price"])*100

In [97]:
sales_clean["stock_gap"] = (
    sales_clean["stock_on_hand"] - sales_clean["reorder_point"]
)

In [98]:
daily_demand=(
    sales_clean.groupby(["date","store_id","sku_id"],as_index=False)["quantity"]
    .sum()
    .rename(columns={"quantity":"daily_quantity"})
)

In [99]:
daily_demand.head()

,date,store_id,sku_id,daily_quantity
0,2022-01-01,ST01,SKU00004,1
1,2022-01-01,ST01,SKU00059,1
2,2022-01-01,ST01,SKU00099,2
3,2022-01-01,ST01,SKU00130,1
4,2022-01-01,ST01,SKU00233,4


In [100]:
daily_demand.shape, daily_demand.columns.tolist()

((8513611, 4), ['date', 'store_id', 'sku_id', 'daily_quantity'])

In [101]:
sku_features = (
    sales_clean[
        ["sku_id", "sku_name", "category", "subcategory", "brand"]
    ]
    .drop_duplicates("sku_id")
)

daily_demand = daily_demand.merge(
    sku_features,
    on="sku_id",
    how="left"
)

In [102]:
daily_demand.head()

,date,store_id,sku_id,daily_quantity,sku_name,category,subcategory,brand
0,2022-01-01,ST01,SKU00004,1,SunriseFoods Eggs Large,Dairy & Bakery,Eggs,SunriseFoods
1,2022-01-01,ST01,SKU00059,1,TrueTaste Butter & Ghee 500ml,Dairy & Bakery,Butter & Ghee,TrueTaste
2,2022-01-01,ST01,SKU00099,2,HealthFirst Footwear Family Pack,Apparel & Footwear,Footwear,HealthFirst
3,2022-01-01,ST01,SKU00130,1,PremiumSelect Pulses & Lentils 250ml,Grocery,Pulses & Lentils,PremiumSelect
4,2022-01-01,ST01,SKU00233,4,CrispKing Writing Instruments 250g,Stationery & Office,Writing Instruments,CrispKing


In [103]:
store_features = (
    sales_clean[
        ["store_id", "store_city", "store_type"]
    ]
    .drop_duplicates("store_id")
)

daily_demand = daily_demand.merge(
    store_features,
    on="store_id",
    how="left"
)

In [104]:
daily_demand.shape

(8513611, 10)

In [105]:
daily_demand = daily_demand.assign(
    year=daily_demand["date"].dt.year,
    month=daily_demand["date"].dt.month,
    day_of_week=daily_demand["date"].dt.dayofweek
)

In [106]:
daily_demand = daily_demand.sort_values(
    ["store_id", "sku_id", "date"]
)

daily_demand["previous_date"] = (
    daily_demand.groupby(["store_id", "sku_id"])["date"].shift(1)
)

daily_demand["lag_1_quantity"] = (
    daily_demand.groupby(["store_id", "sku_id"])["daily_quantity"].shift(1)
)

daily_demand.loc[
    (daily_demand["date"] - daily_demand["previous_date"]).dt.days != 1,
    "lag_1_quantity"
] = 0

In [107]:
daily_demand[
    ["date", "store_id", "sku_id", "daily_quantity", "lag_1_quantity"]
].head(10)

,date,store_id,sku_id,daily_quantity,lag_1_quantity
62577,2022-01-15,ST01,SKU00001,3,0.0
176632,2022-02-09,ST01,SKU00001,4,0.0
412504,2022-03-30,ST01,SKU00001,1,0.0
442752,2022-04-05,ST01,SKU00001,2,0.0
616151,2022-05-09,ST01,SKU00001,3,0.0
631689,2022-05-12,ST01,SKU00001,1,0.0
1298207,2022-09-15,ST01,SKU00001,3,0.0
1404240,2022-10-04,ST01,SKU00001,1,0.0
1661800,2022-11-17,ST01,SKU00001,2,0.0
1802893,2022-12-08,ST01,SKU00001,1,0.0


In [108]:
daily_demand[daily_demand["lag_1_quantity"] > 0][
    ["date", "store_id", "sku_id", "daily_quantity", "lag_1_quantity"]
].head(10)

,date,store_id,sku_id,daily_quantity,lag_1_quantity
4621678,2024-04-18,ST01,SKU00001,1,5.0
5157789,2024-07-20,ST01,SKU00001,4,2.0
8933,2022-01-03,ST01,SKU00002,1,1.0
13426,2022-01-04,ST01,SKU00002,2,1.0
76274,2022-01-18,ST01,SKU00002,1,2.0
213660,2022-02-17,ST01,SKU00002,1,1.0
353213,2022-03-18,ST01,SKU00002,3,1.0
795450,2022-06-12,ST01,SKU00002,3,3.0
817166,2022-06-16,ST01,SKU00002,2,5.0
937159,2022-07-09,ST01,SKU00002,3,1.0


In [109]:
daily_demand = daily_demand.drop(columns="previous_date")

In [110]:
daily_demand["previous_7_date"] = (
    daily_demand.groupby(["store_id", "sku_id"])["date"].shift(1)
)

daily_demand["lag_7_quantity"] = (
    daily_demand.groupby(["store_id", "sku_id"])["daily_quantity"].shift(1)
)

daily_demand.loc[
    (daily_demand["date"] - daily_demand["previous_7_date"]).dt.days != 7,
    "lag_7_quantity"
] = 0

In [111]:
daily_demand[
    daily_demand["lag_7_quantity"] > 0
][
    ["date", "store_id", "sku_id", "daily_quantity", "previous_7_date", "lag_7_quantity"]
].head(10)

,date,store_id,sku_id,daily_quantity,previous_7_date,lag_7_quantity
44481,2022-01-11,ST01,SKU00002,1,2022-01-04,2.0
323492,2022-03-12,ST01,SKU00002,3,2022-03-05,1.0
595560,2022-05-05,ST01,SKU00002,1,2022-04-28,1.0
1513508,2022-10-24,ST01,SKU00002,1,2022-10-17,2.0
1551920,2022-10-31,ST01,SKU00002,3,2022-10-24,1.0
2640534,2023-05-12,ST01,SKU00002,1,2023-05-05,1.0
3295333,2023-09-07,ST01,SKU00002,4,2023-08-31,1.0
3512643,2023-10-14,ST01,SKU00002,3,2023-10-07,1.0
3553173,2023-10-21,ST01,SKU00002,1,2023-10-14,3.0
4150267,2024-01-20,ST01,SKU00002,1,2024-01-13,1.0


In [112]:
daily_demand = daily_demand.drop(columns="previous_7_date")


In [113]:
daily_demand["rolling_7_quantity"] = (
    daily_demand
    .groupby(["store_id", "sku_id"])["daily_quantity"]
    .transform(lambda x: x.shift(1).rolling(7, min_periods=1).mean())
)

In [114]:
daily_demand[
    ["date", "store_id", "sku_id", "daily_quantity",
     "lag_1_quantity", "lag_7_quantity", "rolling_7_quantity"]
].head(10)

,date,store_id,sku_id,daily_quantity,lag_1_quantity,lag_7_quantity,rolling_7_quantity
62577,2022-01-15,ST01,SKU00001,3,0.0,0.0,NaN
176632,2022-02-09,ST01,SKU00001,4,0.0,0.0,3.000000
412504,2022-03-30,ST01,SKU00001,1,0.0,0.0,3.500000
442752,2022-04-05,ST01,SKU00001,2,0.0,0.0,2.666667
616151,2022-05-09,ST01,SKU00001,3,0.0,0.0,2.500000
631689,2022-05-12,ST01,SKU00001,1,0.0,0.0,2.600000
1298207,2022-09-15,ST01,SKU00001,3,0.0,0.0,2.333333
1404240,2022-10-04,ST01,SKU00001,1,0.0,0.0,2.428571
1661800,2022-11-17,ST01,SKU00001,2,0.0,0.0,2.142857
1802893,2022-12-08,ST01,SKU00001,1,0.0,0.0,1.857143


In [115]:
daily_demand["rolling_7_quantity"] = daily_demand["rolling_7_quantity"].fillna(0)

In [116]:
daily_demand["is_weekend"] = (
    daily_demand["day_of_week"] >= 5
).astype(int)

In [117]:
daily_demand["date"].min(), daily_demand["date"].max()

(Timestamp('2022-01-01 00:00:00'), Timestamp('2025-12-31 00:00:00'))

In [118]:
train_data=daily_demand[daily_demand["date"]<"2025-01-01"].copy()

In [119]:
test_data=daily_demand[daily_demand["date"]>="2025-01-01"].copy()

In [120]:
train_data.shape, test_data.shape

((6246038, 17), (2267573, 17))

# data training start

In [121]:
X_train=train_data.drop(columns=["daily_quantity","date"])

In [122]:
X_train.shape,X_train.columns.tolist()

((6246038, 15),
 ['store_id',
  'sku_id',
  'sku_name',
  'category',
  'subcategory',
  'brand',
  'store_city',
  'store_type',
  'year',
  'month',
  'day_of_week',
  'lag_1_quantity',
  'lag_7_quantity',
  'rolling_7_quantity',
  'is_weekend'])

In [123]:
y_train=train_data["daily_quantity"]

In [124]:
y_train.shape

(6246038,)

In [125]:
X_test=test_data.drop(columns=["daily_quantity","date"])

In [126]:
X_test.shape

(2267573, 15)

In [127]:
y_test=test_data["daily_quantity"]

# data saved in csv form so as not to load the file again and again


In [128]:
import os

os.makedirs("../data/processed", exist_ok=True)

daily_demand.to_csv("../data/processed/daily_demand.csv", index=False)
train_data.to_csv("../data/processed/train_data.csv", index=False)
test_data.to_csv("../data/processed/test_data.csv", index=False)